In [1]:
# Day 10: GenAI Insight Engine with Gemini
# ==========================================
import pandas as pd
import google.generativeai as gemini
from sqlalchemy import create_engine, text
import os
import json
from dotenv import load_dotenv
from datetime import datetime

load_dotenv('../.env')
warehouse_connection = create_engine(os.getenv('DATABASE_URL'))
gemini.configure(api_key=os.getenv('GEMINI_API_KEY'))
ai_model = gemini.GenerativeModel('gemini-flash-latest')
print("Connected to warehouse and Gemini AI")

# ===== GATHER BUSINESS METRICS FROM WAREHOUSE =====
print("\nGathering business metrics...")

funnel_query = """
WITH user_journey AS (
    SELECT user_id,
        MAX(CASE WHEN event_type = 'page_view' THEN 1 ELSE 0 END) AS viewed,
        MAX(CASE WHEN event_type = 'product_view' THEN 1 ELSE 0 END) AS product_viewed,
        MAX(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS carted,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchased
    FROM fact_events
    GROUP BY user_id
)
SELECT 
    SUM(viewed) AS total_viewers,
    SUM(product_viewed) AS total_product_viewers,
    SUM(carted) AS total_cart_adders,
    SUM(purchased) AS total_buyers,
    ROUND(100.0 * SUM(purchased) / NULLIF(SUM(viewed), 0), 2) AS overall_conversion_rate
FROM user_journey;
"""

revenue_query = """
SELECT 
    COUNT(*) AS total_purchases,
    ROUND(SUM(amount)::numeric, 2) AS total_revenue,
    ROUND(AVG(amount)::numeric, 2) AS avg_order_value,
    ROUND(MAX(amount)::numeric, 2) AS max_purchase
FROM fact_events
WHERE event_type = 'purchase' AND amount IS NOT NULL;
"""

business_metrics = pd.read_sql(funnel_query, warehouse_connection).iloc[0].to_dict()
revenue_metrics = pd.read_sql(revenue_query, warehouse_connection).iloc[0].to_dict()

# Load RFM segments from Day 9
rfm_results = pd.read_csv('../data/processed/rfm_segmentation.csv')
segment_distribution = rfm_results['customer_segment'].value_counts().to_dict()

business_snapshot = {
    'funnel': {k: int(v) if pd.notna(v) else 0 for k, v in business_metrics.items() if k != 'overall_conversion_rate'},
    'conversion_rate_pct': float(business_metrics['overall_conversion_rate']),
    'revenue': {k: float(v) if pd.notna(v) else 0 for k, v in revenue_metrics.items()},
    'customer_segments': {k: int(v) for k, v in segment_distribution.items()}
}

print("\nBusiness Snapshot:")
print(json.dumps(business_snapshot, indent=2))

# ===== GENERATE AI-POWERED EXECUTIVE SUMMARY =====
print("\n" + "="*60)
print("GENERATING AI EXECUTIVE SUMMARY...")
print("="*60)

executive_prompt = f"""
You are a senior data analyst at an e-commerce company. 
Analyze the business metrics below and write a 3-paragraph executive summary.

BUSINESS METRICS:
{json.dumps(business_snapshot, indent=2)}

Your summary should include:
1. Top 3 key findings with specific numbers
2. The biggest problem area and its root cause
3. 3 prioritized action items with estimated revenue impact

Keep it concise, data-driven, and actionable. Write for a CEO audience.
"""

executive_response = ai_model.generate_content(executive_prompt)
executive_summary = executive_response.text
print(executive_summary)

# ===== GENERATE SEGMENT-SPECIFIC RECOMMENDATIONS =====
print("\n" + "="*60)
print("GENERATING SEGMENT RECOMMENDATIONS...")
print("="*60)

segment_prompt = f"""
You are an e-commerce marketing strategist.
Based on this customer segmentation, write specific marketing 
recommendations for EACH segment. Be practical and actionable.

CUSTOMER SEGMENTS:
{json.dumps(segment_distribution, indent=2)}

For each segment, provide:
- Campaign objective (1 sentence)
- Recommended channel (email, SMS, ads, etc.)
- Sample message copy (1-2 sentences)
- Expected outcome

Format as: SEGMENT NAME followed by recommendations.
"""

segment_response = ai_model.generate_content(segment_prompt)
segment_recommendations = segment_response.text
print(segment_recommendations)

# ===== NATURAL LANGUAGE TO SQL =====
print("\n" + "="*60)
print("TESTING NATURAL LANGUAGE TO SQL...")
print("="*60)

def ask_question_in_english(user_question):
    """Convert a plain English question to SQL and execute it."""
    
    sql_prompt = f"""
You are a SQL expert. Convert this question into a PostgreSQL query.

DATABASE SCHEMA:
- fact_events: event_id, user_id, session_id, timestamp, event_type, product_id, amount
  - event_type values: 'page_view', 'product_view', 'click', 'add_to_cart', 'purchase', 'login', 'logout'
- dim_users: user_id, first_seen, last_seen, total_events
- dim_products: product_id, product_key
- dim_dates: date, year, month, day, quarter, day_of_week

QUESTION: {user_question}

Return ONLY the SQL query, no explanation, no markdown code blocks, no semicolon at end.
"""
    
    sql_response = ai_model.generate_content(sql_prompt)
    generated_sql = sql_response.text.strip()
    generated_sql = generated_sql.replace('```sql', '').replace('```', '').strip()
    
    print(f"\nQuestion: {user_question}")
    print(f"Generated SQL:\n{generated_sql}\n")
    
    try:
        result_df = pd.read_sql(generated_sql, warehouse_connection)
        print(f"Results:\n{result_df.head(10)}")
        return result_df
    except Exception as e:
        print(f"Query failed: {e}")
        return None

sample_questions = [
    "What is the total revenue from all purchases?",
    "How many unique users made a purchase in 2025?",
    "What are the top 5 hours of the day by number of purchases?"
]

for question in sample_questions:
    print("\n" + "-"*60)
    ask_question_in_english(question)

# ===== SAVE AI OUTPUTS =====
os.makedirs('../docs/ai_insights', exist_ok=True)

with open('../docs/ai_insights/executive_summary.txt', 'w') as f:
    f.write("EXECUTIVE SUMMARY (AI-Generated)\n")
    f.write("="*60 + "\n")
    f.write(f"Generated: {datetime.now()}\n\n")
    f.write(executive_summary)

with open('../docs/ai_insights/segment_recommendations.txt', 'w') as f:
    f.write("SEGMENT RECOMMENDATIONS (AI-Generated)\n")
    f.write("="*60 + "\n")
    f.write(f"Generated: {datetime.now()}\n\n")
    f.write(segment_recommendations)

with open('../docs/ai_insights/business_snapshot.json', 'w') as f:
    json.dump(business_snapshot, f, indent=2)

print("\n" + "="*60)
print("AI insights saved to docs/ai_insights/")
print("Day 10 complete!")

/var/folders/83/8k0mgq8n5c74kg5014bjrd300000gn/T/ipykernel_1877/3534491304.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as gemini


Connected to warehouse and Gemini AI

Gathering business metrics...

Business Snapshot:
{
  "funnel": {
    "total_viewers": 4870,
    "total_product_viewers": 4806,
    "total_cart_adders": 1865,
    "total_buyers": 905
  },
  "conversion_rate_pct": 18.58,
  "revenue": {
    "total_purchases": 1001.0,
    "total_revenue": 255375.66,
    "avg_order_value": 255.12,
    "max_purchase": 499.92
  },
  "customer_segments": {
    "Loyal Customers": 173,
    "Champions": 166,
    "Lost": 155,
    "Needs Attention": 153,
    "New Customers": 135,
    "At Risk": 123
  }
}

GENERATING AI EXECUTIVE SUMMARY...
**Executive Summary: E-Commerce Performance Analysis**

Our current performance reflects a healthy overall conversion rate of 18.58%, generating $255,375.66 in total revenue from 1,001 purchases. Three key findings stand out: first, our Average Order Value (AOV) is robust at $255.12, indicating strong product positioning; second, our customer base is anchored by 339 "Champions" and "Loyal Cu